# Selección de variables: SelectKBest (ANOVA F), RFE y Lasso

El pipeline de preprocesamiento deja cerca de 500 variables candidatas: cierres diarios, rezagos, medias móviles, ratios y clima. Con tantas variables los modelos se sobreajustan y Optuna explora un espacio innecesariamente grande. La consigna (2.3) pide aplicar al menos dos métodos de selección y justificar el conjunto final. Acá corremos tres y elegimos uno con criterios medidos.

**Métodos comparados**

| Método | Tipo | Cómo ordena las variables |
|---|---|---|
| `SelectKBest(f_regression)` | Filtro | Estadístico F del ANOVA de una regresión lineal simple de cada variable contra el target. Es la versión de ANOVA para un target continuo. |
| `RFE` con Random Forest | Envoltura | Entrena un bosque, descarta el 5 % de variables menos importantes y repite hasta quedarse con una. |
| `LassoCV` | Embebido | Regresión lineal con penalización L1 y alpha elegido con `TimeSeriesSplit`; ordena por el valor absoluto del coeficiente y descarta los nulos. |

**Protocolo (sin fuga de información)**

- Separamos las fechas elegibles en Train, Validation y Test (70/15/15). El experimento usa las fechas de Train y Validation.
- Dentro de Train corremos una validación cruzada temporal (cinco folds de 100 observaciones). En cada fold se ajustan con su propio entrenamiento la auditoría de sensores, los filtros de columnas, la imputación, los límites de atípicos, el escalado y el ranking de variables.
- Después ajustamos todo con Train completo y medimos una sola vez en Validation.
- Cada subconjunto se evalúa con dos modelos, Ridge (lineal) y LightGBM (árboles), para no favorecer a un método que solo le sirva a un tipo de modelo. Probamos k = 5, 10, 15, 20, 30, 40, 60 y 80 variables, y también todas.
- Medimos la estabilidad de cada método: el índice de Jaccard entre los conjuntos que elige en los distintos folds (1 = siempre las mismas variables).
- Referencias: los baselines de persistencia (`ŷ[D+1] = y[D]`) y de media móvil de 7 días (promedio de los siete días anteriores a `D+1`), calculados con `baselines_validacion` de `src/evaluate_model.py` sobre los mismos días.

In [ ]:
import json
import sys
import time
import warnings
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import RFE, f_regression
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LassoCV, Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit

from pandas.errors import PerformanceWarning

# Avisos de rendimiento del pipeline y de columnas vacías dentro de algunos folds: no cambian los resultados
for categoria in (UserWarning, RuntimeWarning, PerformanceWarning):
    warnings.filterwarnings('ignore', category=categoria)
RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(RAIZ / 'src'))
import preprocessing_pipeline as pp
from evaluate_model import baselines_validacion

SEMILLA = 42
KS = [5, 10, 15, 20, 30, 40, 60, 80]
METODOS = ['SelectKBest (ANOVA F)', 'RFE (Random Forest)', 'Lasso']
CARPETA_RESULTADOS = RAIZ / 'results' / 'seleccion_variables'
CARPETA_RESULTADOS.mkdir(parents=True, exist_ok=True)

## 1. Datos del pipeline

Definimos las fechas antes de ajustar filtros o imputadores. Cargamos el clima una vez para el período de desarrollo.

In [ ]:
ruta_unificado = RAIZ / 'data' / 'dataset_unificado_inicial.csv'
df_diario = pp.cargar_y_limpiar_registros(ruta_unificado)
fechas_train, fechas_val, _ = pp.dividir_temporalmente(pp.fechas_modelables(df_diario))
df_diario = df_diario.loc[:fechas_val[-1]]
clima = pp.cargar_clima(fechas_train.append(fechas_val))
y_train = df_diario.loc[fechas_train, pp.TARGET]
y_val = df_diario.loc[fechas_val, pp.TARGET]

print(f'Train: {len(fechas_train)} días ({fechas_train[0]:%Y-%m-%d} a {fechas_train[-1]:%Y-%m-%d})')
print(f'Validation: {len(fechas_val)} días ({fechas_val[0]:%Y-%m-%d} a {fechas_val[-1]:%Y-%m-%d})')

## 2. Funciones de la comparación

- `preparar` ajusta el mismo escalado del pipeline y una imputación por mediana con los datos de entrenamiento del fold. Acotamos los valores escalados a ±8 desvíos: algunos sensores cambian de escala entre períodos y, sin ese límite, Ridge extrapola y da errores de millones de kW que no dicen nada del método de selección.
- `ordenar_variables` devuelve, para cada método, las variables de la más a la menos relevante. En RFE desempatamos las variables descartadas en el mismo paso con la importancia del bosque completo.
- `evaluar` entrena Ridge y LightGBM con las primeras `k` variables de cada orden y mide el MAE.

In [ ]:
def preparar(fechas_ajuste, fechas_evaluacion):
    diario = pp.auditar_sensores_muertos(
        df_diario.loc[:fechas_evaluacion[-1]], hasta=fechas_ajuste[-1],
    )
    diario = pp.crear_features(diario, clima=clima)
    X_ajuste, X_evaluacion = pp.filtrar_imputar_y_limpiar(
        diario.loc[fechas_ajuste], diario.loc[fechas_evaluacion],
    )
    escalado = pp.construir_pipeline_escalado(X_ajuste)
    a = escalado.fit_transform(X_ajuste).replace([np.inf, -np.inf], np.nan)
    b = escalado.transform(X_evaluacion).replace([np.inf, -np.inf], np.nan)
    imputador = SimpleImputer(strategy='median', keep_empty_features=True).set_output(transform='pandas')
    a = imputador.fit_transform(a)
    b = imputador.transform(b)[a.columns]
    return a.clip(-8, 8), b.clip(-8, 8)


def ordenar_variables(X, y):
    f, _ = f_regression(X, y)
    orden_kbest = pd.Series(np.nan_to_num(f), index=X.columns).sort_values(ascending=False).index.tolist()

    rfe = RFE(RandomForestRegressor(n_estimators=150, random_state=SEMILLA, n_jobs=-1),
              n_features_to_select=1, step=0.05).fit(X, y)
    importancia = RandomForestRegressor(n_estimators=150, random_state=SEMILLA, n_jobs=-1).fit(X, y).feature_importances_
    orden_rfe = (pd.DataFrame({'paso': rfe.ranking_, 'importancia': -importancia}, index=X.columns)
                 .sort_values(['paso', 'importancia']).index.tolist())

    lasso = LassoCV(cv=TimeSeriesSplit(5), random_state=SEMILLA, max_iter=50000, n_jobs=-1).fit(X, y)
    coeficientes = pd.Series(np.abs(lasso.coef_), index=X.columns)
    orden_lasso = coeficientes[coeficientes > 0].sort_values(ascending=False).index.tolist()
    return dict(zip(METODOS, [orden_kbest, orden_rfe, orden_lasso])), lasso.alpha_


def modelos():
    return {
        'Ridge': Ridge(alpha=1.0),
        'LightGBM': LGBMRegressor(n_estimators=400, learning_rate=0.03, num_leaves=15,
                                  random_state=SEMILLA, n_jobs=-1, verbosity=-1),
    }


def evaluar(X_a, y_a, X_b, y_b, ordenes, etiqueta):
    filas = []
    conjuntos = [('Todas', list(X_a.columns))]
    for metodo, orden in ordenes.items():
        conjuntos += [(metodo, orden[:k]) for k in KS if len(orden) >= k]
    if ordenes['Lasso']:
        conjuntos.append(('Lasso (alpha por CV)', ordenes['Lasso']))
    for metodo, columnas in conjuntos:
        for nombre, modelo in modelos().items():
            modelo.fit(X_a[columnas], y_a)
            filas.append({'evaluacion': etiqueta, 'metodo': metodo, 'k': len(columnas), 'modelo': nombre,
                          'MAE': mean_absolute_error(y_b, modelo.predict(X_b[columnas]))})
    return filas

## 3. Validación cruzada temporal dentro de Train y medición en Validation

In [ ]:
resultados, ordenes_por_fold, baselines = [], [], []
particion = TimeSeriesSplit(n_splits=5, test_size=100)
for numero, (idx_a, idx_b) in enumerate(particion.split(fechas_train), start=1):
    inicio = time.time()
    X_a, X_b = preparar(fechas_train[idx_a], fechas_train[idx_b])
    ordenes, alpha = ordenar_variables(X_a, y_train.iloc[idx_a])
    ordenes_por_fold.append(ordenes)
    resultados += evaluar(X_a, y_train.iloc[idx_a], X_b, y_train.iloc[idx_b], ordenes, f'CV fold {numero}')
    baselines.append(baselines_validacion(y_train.iloc[idx_a], y_train.iloc[idx_b])['MAE']
                     .rename(f'CV fold {numero}'))
    print(f'Fold {numero}: entrena {len(idx_a)} días, evalúa {len(idx_b)} | Lasso alpha = {alpha:.0f}, '
          f'variables no nulas = {len(ordenes["Lasso"])} | {time.time() - inicio:.0f} s')

X_tr, X_va = preparar(fechas_train, fechas_val)
ordenes_train, alpha_train = ordenar_variables(X_tr, y_train)
resultados += evaluar(X_tr, y_train, X_va, y_val, ordenes_train, 'Validation')
baselines.append(baselines_validacion(y_train, y_val)['MAE'].rename('Validation'))
resultados = pd.DataFrame(resultados)
baselines = pd.DataFrame(baselines).rename_axis('evaluacion')
print(f'Train completo: Lasso alpha = {alpha_train:.0f}, variables no nulas = {len(ordenes_train["Lasso"])}')

## 4. Resultados

MAE en kW. Una configuración tiene media y desvío de CV cuando se pudo evaluar en los cinco folds. La columna Folds indica la cobertura de cada configuración.

In [ ]:
cv = resultados[resultados['evaluacion'].str.startswith('CV')]
resumen_cv = cv.groupby(['metodo', 'k', 'modelo'])['MAE'].agg(['mean', 'std', 'count']).unstack('modelo')
resumen_val = (resultados[resultados['evaluacion'].eq('Validation')]
               .pivot_table(index=['metodo', 'k'], columns='modelo', values='MAE'))
faltan_folds = resumen_cv['count'].ne(particion.n_splits)
media_cv = resumen_cv['mean'].mask(faltan_folds)
desvio_cv = resumen_cv['std'].mask(faltan_folds)
tabla = pd.concat({'CV media': media_cv, 'CV desvío': desvio_cv,
                   'Folds': resumen_cv['count'], 'Validation': resumen_val}, axis=1)
tabla = tabla[tabla.index.get_level_values('metodo') != 'Lasso (alpha por CV)']
display(tabla.round(0))

print('Baselines (MAE)')
display(pd.DataFrame({'CV media': baselines.iloc[:-1].mean(), 'Validation': baselines.loc['Validation']}).round(0))

lasso_cv = resultados[resultados['metodo'].eq('Lasso (alpha por CV)')]
print('Lasso con su propio alpha: cantidad de variables elegidas en cada evaluación')
display(lasso_cv.pivot_table(index=['evaluacion', 'k'], columns='modelo', values='MAE').round(0))

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True, layout='constrained')
colores = {'SelectKBest (ANOVA F)': '#2a6f97', 'RFE (Random Forest)': '#c0703a', 'Lasso': '#5a8f4e'}
for eje, (titulo, datos) in zip(ejes, [('Validación cruzada temporal (media de 5 folds)', media_cv),
                                       ('Validation', resumen_val)]):
    for metodo, color in colores.items():
        serie = datos.reindex([(metodo, k) for k in KS])
        posiciones = range(len(KS))
        eje.plot(posiciones, serie['Ridge'], marker='o', color=color, label=f'{metodo} + Ridge')
        eje.plot(posiciones, serie['LightGBM'], marker='o', linestyle=':', color=color, alpha=.6,
                 label=f'{metodo} + LightGBM')
    referencia = baselines.iloc[:-1].mean() if titulo.startswith('Validación') else baselines.loc['Validation']
    eje.axhline(referencia['Persistencia'], color='#555', linestyle='--', linewidth=1, label='Persistencia')
    eje.axhline(referencia['Media móvil 7 días'], color='#999', linestyle='--', linewidth=1, label='Media móvil 7 días')
    eje.set(title=titulo, xlabel='Cantidad de variables (k)', xticks=range(len(KS)), xticklabels=KS)
    eje.grid(alpha=.25)
ejes[0].set_ylabel('MAE (kW)')
ejes[0].set_ylim(1500, 6000)
ejes[1].legend(fontsize=8, loc='upper left', bbox_to_anchor=(1.01, 1))
plt.show()

### Estabilidad de la selección

Para cada método y cada k comparamos los conjuntos elegidos en los cinco folds de a pares y promediamos el índice de Jaccard (variables en común sobre variables en total).

In [ ]:
estabilidad = []
for metodo in METODOS:
    for k in [5, 10, 20, 30]:
        conjuntos = [set(orden[metodo][:k]) for orden in ordenes_por_fold if len(orden[metodo]) >= k]
        jaccard = np.nan
        if len(conjuntos) == particion.n_splits:
            pares = [len(a & b) / len(a | b) for i, a in enumerate(conjuntos) for b in conjuntos[i + 1:]]
            jaccard = np.mean(pares)
        estabilidad.append({'metodo': metodo, 'k': k, 'Jaccard medio': jaccard})
estabilidad = pd.DataFrame(estabilidad)
display(estabilidad.pivot(index='metodo', columns='k', values='Jaccard medio').round(2))

### Variables elegidas con Train completo

In [ ]:
top = pd.DataFrame({metodo: pd.Series(orden[:15]) for metodo, orden in ordenes_train.items()})
top.index = top.index + 1
display(top)
coincidencias = {f'{a} y {b}': len(set(ordenes_train[a][:10]) & set(ordenes_train[b][:10]))
                 for i, a in enumerate(METODOS) for b in METODOS[i + 1:]}
print('Variables en común entre los top 10:', coincidencias)

## 5. Evaluación del conjunto de 10 variables

El pipeline usa ANOVA F con k = 10. Para revisar esa elección, compará primero el MAE medio y su dispersión entre los folds temporales. Después contrastá con el MAE en Validation y la estabilidad de las variables entre folds. Si la diferencia de MAE es chica frente a la dispersión, el tamaño del conjunto y su estabilidad ayudan a decidir.

El ranking de ANOVA F mide la relación individual de cada variable con el target. Compararlo con RFE y Lasso permite ver si otras formas de seleccionar variables mejoran el MAE de Ridge o LightGBM.

## 6. Registro de la comparación

Guardamos todos los resultados en `results/seleccion_variables/` para que la decisión quede trazable:

- `resultados.csv`: MAE de cada método, k, modelo y evaluación (cada fold y Validation).
- `baselines.csv` y `estabilidad.csv`.
- `seleccion_final.json`: método elegido, k, variables seleccionadas con Train completo y las de los otros métodos.

In [ ]:
K_ELEGIDO = pp.K_VARIABLES
variables_finales = ordenes_train['SelectKBest (ANOVA F)'][:K_ELEGIDO]
puntaje_f, p_valor = f_regression(X_tr[variables_finales], y_train)

resultados.to_csv(CARPETA_RESULTADOS / 'resultados.csv', index=False)
baselines.to_csv(CARPETA_RESULTADOS / 'baselines.csv')
estabilidad.to_csv(CARPETA_RESULTADOS / 'estabilidad.csv', index=False)
registro = {
    'fecha': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'metodo_elegido': 'SelectKBest(f_regression)',
    'k': K_ELEGIDO,
    'ajustado_con': 'Train',
    'variables_seleccionadas': variables_finales,
    'metodos_comparados': {metodo: orden[:K_ELEGIDO] for metodo, orden in ordenes_train.items()},
    'lasso_alpha_train': float(alpha_train),
    'lasso_variables_no_nulas_train': len(ordenes_train['Lasso']),
}
(CARPETA_RESULTADOS / 'seleccion_final.json').write_text(json.dumps(registro, indent=2, ensure_ascii=False) + '\n',
                                                          encoding='utf-8')
display(pd.DataFrame({'F': puntaje_f.round(0), 'p': p_valor}, index=variables_finales))
print(f'Registro guardado en {CARPETA_RESULTADOS.relative_to(RAIZ).as_posix()}/')